# Prompt Rescue ワークショップ

## ビルド開始から3週間。精度19%。明日レビューがある。

あなたのクライアント、**TechSupport Corp** は Claude を使ったサポートチケット処理ツールを構築しました — P1〜P4 の優先度分類、エンティティ抽出、回答ドラフトを1つのプロンプトで行います。クリーンなチケットでは動作しますが、実際の入力では崩壊します: 優先度の誤分類、幻覚エンティティ、壊れた JSON、一貫性のない回答。

明日のレビューまでに — モデルを交換しようという提案が出る前に — 何が本当に壊れているかを把握する必要があります。それがこのセッションです。

**あなたの仕事:** 失敗を診断し、問題を修正し、自分の判断を明確に説明できるようにする。説明できないスコア向上はデモに過ぎません。診断に裏付けられたものが方法論です。

### 学習目標

このワークショップを終える頃には、次のことができるようになります:

1. **診断する** プロンプト失敗の根本原因を — 症状だけでなく、*構造的な* 理由を
2. **適用する** 各選択の根拠を明確にしながらプロンプトエンジニアリングの技術を
3. **評価する** 定量的な評価ハーネスで改善を測定する — 感覚頼みの評価ではなく
4. **説明する** 評価結果を根拠に設計上の判断を
5. **振り返る** 自分がエンジニアリングしたものと Claude が行ったこと — クライアントとの会話で重要な区別

### 形式

これは **個人演習** です。以下のセルを自分のペースで進めてください。

**推奨時間配分:** 診断 〜15分 → 構築 〜45分 → 振り返り 〜10分

In [ ]:
# Anthropic SDK のインストール
!pip install -q anthropic matplotlib --upgrade

# APIキーの設定
import os
ANTHROPIC_API_KEY = ""  # <-- ここにAPIキーを貼り付けてください
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

# 接続確認
import anthropic
client = anthropic.Anthropic()
test = client.messages.create(
    model="claude-haiku-4-5",
    max_tokens=50,
    messages=[{"role": "user", "content": "Say 'Connected!' and nothing else."}]
)
print(f"API Connected: {test.content[0].text}")

## シナリオ: TechSupport Corp

### 顧客概要

> "私たちは TechSupport Corp、エンタープライズ SaaS 企業です。受信サポートチケットを処理するプロンプトを構築しました — 優先度の分類、主要な詳細の抽出、回答のドラフトをワンショットで行います。シンプルで明確に書かれたチケットでは完璧に動きます。しかし本番環境では、チケットの約半数が雑然としています — 複数の問題、曖昧な説明、英語が母国語でない話者のもの。それらがひどく失敗します。JSON が壊れて返ってくる、優先度が間違っている、ドラフトした回答が分類と矛盾することもある。更新ミーティングの前に修正が必要です。制約として: 私たちのチケット処理パイプラインは1チケットあたり5秒のレイテンシ予算があるため、3回以上の順次APIコールを要するソリューションには正当化が必要です。"

### 課題

1. **診断する** 失敗の根本原因（症状だけでなく — プロンプト構造が*なぜ*これらの失敗を生み出すのか？）
2. **修正する** ツールキットから少なくとも2つの技術を使ってプロンプトを改善する（「ツールキット」の項を参照）
3. **検証する** 評価スイートで修正を確認する
4. **発表する** 3分間の「顧客デブリーフ」 — 診断、修正、結果、予防のアドバイス

### 成功基準

| レベル | 目標 |
|-------|--------|
| **ベースライン（合格）** | 評価パス率75%以上、根本原因の特定、2つ以上の技術を根拠とともに使用、一貫したデブリーフ |
| **Tier 1 — オプティマイザー** | パス率90%以上、ドキュメント化されたイテレーションログ、モニタリングプロンプトの設計 |
| **Tier 2 — アーキテクト** | セルフヒーリングプロンプトチェーン（品質チェック失敗時の自動リトライ） |

### 制約

- **モデル:** Claude Haiku（評価ハーネスにハードコード済み — モデルアップグレードは修正として使えない）
- **アーキテクチャ:** プロンプトを完全に再構成できます — 複数ステップやチェーンへの分解も含めて — ただしエンドツーエンドの機能は同じでなければなりません: 同じ入力、同じタイプの出力
- **プロセス:** 各プロンプトのイテレーションを記録してください。評価ハーネスが自動的に進捗を記録します。

In [ ]:
# ============================================================
# あなたのプロンプト -- これを編集して壊れたプロンプトを修正してください！
# ============================================================

system_prompt = """
You are a support ticket processor. For each ticket, you must:
1. Classify priority (P1-P4) based on business impact
2. Extract: product name, version, error codes, affected users count
3. Draft a helpful response acknowledging the issue and providing next steps
4. Return everything as JSON: {"priority": "", "entities": {"product": "", "version": "", "error_codes": [], "affected_users": ""}, "response": "", "confidence": "high/medium/low"}

Rules:
- P1 = system down, all users affected
- P2 = major feature broken, many users affected
- P3 = minor bug, few users affected
- P4 = feature request or cosmetic issue
- If unsure about priority, use your best judgment
- Response should be professional and empathetic
- Always include all JSON fields even if empty
- Be concise but thorough
"""

print("壊れたプロンプトを読み込みました。評価スイート（セル7）を実行してベースラインスコアを確認してください。")
print(f"プロンプトの長さ: {len(system_prompt)} 文字")

## 失敗例 — 診断資料

これらは壊れたプロンプトが失敗する5件の実際の入力です。修正を始める前にパターンを分析してください。

---

### 例1: 複数問題チケット

**入力:**
```
Subject: URGENT - billing broken AND can't export data

Our billing dashboard has been showing wrong numbers since Tuesday and also the CSV export
feature throws a 500 error. We have board reporting on Friday and need both fixed. About 200
users on our team are affected by the export issue, billing is just our finance team (3 people)
but it's blocking payroll.
```

**何が起きたか:** P2 と分類している（正しくは P1 — 給与計算がブロックされている）。両方の問題を1つのエンティティ抽出にまとめてしまっている。回答はエクスポートの問題のみ扱い、請求・給与の問題を無視している。

---

### 例2: 曖昧なチケット

**入力:**
```
Subject: things arent working right

hey so a bunch of stuff seems off today? like the pages load slow and sometimes i get errors.
not sure whats happening. its been like this since the morning. thanks
```

**何が起きたか:** 入力のどこにも存在しないプロダクト名「PageLoader」やエラーコード「ERR_SLOW_LOAD」を幻覚している。確信を持って分類できる情報が不足しているのに P3 と分類している。

---

### 例3: 英語が母国語でない話者

**入力:**
```
Subject: Probleme mit der Integration

Hello, we have problem with API integration since update to version 3.2. The webhook is not
fire correctly and data is missing in dashboard. Very urgent for us because is production
system for 50 customer. Error say "timeout exceeded" sometimes.
```

**何が起きたか:** JSON 出力が不正な形式になっている — response フィールドに引用符が含まれており、JSON 構造を壊している。優先度は正しい（P2）が、エンティティがバージョン番号を見落としている。

---

### 例4: バグに偽装した機能要望

**入力:**
```
Subject: BUG: No dark mode support

This is a critical usability bug. Your application doesn't support dark mode which causes eye
strain for our entire team working night shifts (15 people). This should have been implemented
long ago. We need this resolved immediately or we'll consider switching providers.
```

**何が起きたか:** 緊急を示す言葉のせいで P2 と分類しているが、明らかに P4（機能要望）である。回答が「バグを修正する」と約束しており、誤った顧客期待を生んでいる。

---

### 例5: クリーンなチケット（リグレッションチェック）

**入力:**
```
Subject: Error 503 on login page

Since 2pm EST today, all users (approximately 500) in our organization are unable to log into
the platform. We receive a 503 Service Unavailable error. This is our primary CRM tool and
sales team is completely blocked. Error code: SVC-503-AUTH. Running version 4.1.2.
```

**何が起きたか:** ほとんどの場合これを正しく処理できる（クリーンな入力）。修正が簡単なケースでリグレッションしないことを確認するために含めています。

In [ ]:
#@title 評価ハーネス（クリックして読み込む -- このセルは編集しないこと）

# ============================================================
# 評価ハーネス -- すべての評価ロジックはここに組み込まれています。
# このセルを1度実行してハーネス関数を読み込んでください。
# ============================================================

import json
import re
import base64
import time
from datetime import datetime
import anthropic
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import display, HTML
matplotlib.rcParams['figure.facecolor'] = 'white'

# ---------------------------------------------------------------------------
# 組み込みの評価ケース（気軽に覗けないよう base64 エンコード済み）
# ---------------------------------------------------------------------------

_EVAL_CASES_B64 = "..."

# ---------------------------------------------------------------------------
# グローバル状態
# ---------------------------------------------------------------------------

_iteration_log = []
_baseline_score = None
_baseline_total = None

MODEL = "claude-haiku-4-5"
MAX_TOKENS = 4096

# ---------------------------------------------------------------------------
# ケースローダー
# ---------------------------------------------------------------------------

def _load_cases(encoded):
    return json.loads(base64.b64decode(encoded).decode("utf-8"))

# ---------------------------------------------------------------------------
# スコアリングモジュール（scoring.py から移植）
# ---------------------------------------------------------------------------

REQUIRED_FIELDS = {"priority", "entities", "response", "confidence"}
REQUIRED_ENTITY_FIELDS = {"product", "version", "error_codes", "affected_users"}


def parse_output(raw_output):
    """モデル出力からJSONを抽出する。ネストされたオブジェクトにも対応。"""
    try:
        parsed = json.loads(raw_output.strip())
        return parsed, None
    except json.JSONDecodeError:
        pass
    start_indices = [i for i, c in enumerate(raw_output) if c == '{']
    for start in start_indices:
        brace_count = 0
        for end in range(start, len(raw_output)):
            if raw_output[end] == '{':
                brace_count += 1
            elif raw_output[end] == '}':
                brace_count -= 1
            if brace_count == 0:
                candidate = raw_output[start:end + 1]
                try:
                    parsed = json.loads(candidate)
                    return parsed, None
                except json.JSONDecodeError:
                    break
    return None, "Output is not valid JSON"


def check_valid_json(parsed, error):
    if parsed is None:
        return False, error or "Output is not valid JSON"
    missing_top = REQUIRED_FIELDS - set(parsed.keys())
    if missing_top:
        return False, f"Missing required fields: {', '.join(sorted(missing_top))}"
    entities = parsed.get("entities")
    if not isinstance(entities, dict):
        return False, "Field 'entities' must be a JSON object"
    missing_entity = REQUIRED_ENTITY_FIELDS - set(entities.keys())
    if missing_entity:
        return False, f"Missing entity fields: {', '.join(sorted(missing_entity))}"
    return True, "Valid JSON with all required fields"


def check_priority(parsed, gold_priority):
    if parsed is None:
        return False, "Cannot check priority -- invalid JSON"
    output_priority = str(parsed.get("priority", "")).strip().upper()
    expected = gold_priority.strip().upper()
    if output_priority == expected:
        return True, f"Priority correct: {expected}"
    return False, f"Priority={output_priority}, expected={expected}"


def _normalize_value(val):
    return re.sub(r'[\s\-_\.]+', ' ', str(val).lower().strip())


_NULL_EQUIVALENT_PATTERN = re.compile(
    r'^('
    r'null|none|n/?a|unknown|unspecified|not\s+specified'
    r'|not\s+provided|not\s+mentioned|not\s+available'
    r'|unknown\s*[-\u2014\u2013]\s*not\s+\w+'
    r'|\w+\s*\(unspecified[^)]*\)'
    r')$',
    re.IGNORECASE,
)


def _is_null_equivalent(val_str):
    """値が意味的に null（モデルが「存在しない」と言っている）かどうかを返す。"""
    return bool(_NULL_EQUIVALENT_PATTERN.match(val_str.strip()))


def _strip_annotations(val):
    """括弧内の接尾辞と先頭の近似マーカーを除去する。"""
    val = re.sub(r'\s*\(.*?\)\s*$', '', val)
    val = re.sub(r'^[~\u2248]\s*', '', val)
    return val.strip()


def _value_in_input(value, input_text):
    """エンティティの値が入力テキストから導出可能かどうかを確認する。"""
    if value is None or value == "" or value == []:
        return True
    input_lower = input_text.lower()
    input_normalized = _normalize_value(input_text)
    if isinstance(value, list):
        return all(_value_in_input(v, input_text) for v in value)
    val_str = str(value).strip()
    if not val_str or _is_null_equivalent(val_str):
        return True
    val_lower = val_str.lower()
    val_normalized = _normalize_value(val_str)
    if val_lower in input_lower:
        return True
    if val_normalized in input_normalized:
        return True
    if re.match(r'^\d+$', val_str):
        if re.search(r'\b' + re.escape(val_str) + r'\b', input_text):
            return True
    if re.match(r'^\d+\.\d+(\.\d+)?$', val_str):
        if val_str in input_text:
            return True
    # フォールバック: アノテーションを除去して再試行
    stripped = _strip_annotations(val_str)
    if stripped and stripped != val_str:
        if _value_in_input(stripped, input_text):
            return True
    # フォールバック: 入力中の近似表現を持つ数値
    num_match = re.match(r'^[~\u2248]?\s*(\d+(?:\.\d+)?)\s*$', val_str)
    if num_match:
        number = num_match.group(1)
        approx_pattern = (
            r'(?:approximately|about|around|roughly|~|\u2248)\s*'
            + re.escape(number)
        )
        if re.search(approx_pattern, input_text, re.IGNORECASE):
            return True
        if re.search(r'\b' + re.escape(number) + r'\b', input_text):
            return True
    # フォールバック: カンマ区切りの複数値
    if ',' in val_str:
        parts = [p.strip() for p in val_str.split(',') if p.strip()]
        if len(parts) > 1 and all(
            _value_in_input(part, input_text) for part in parts
        ):
            return True
    return False


def check_entities(parsed, input_text, gold_entities):
    if parsed is None:
        return False, "Cannot check entities -- invalid JSON"
    entities = parsed.get("entities", {})
    if not isinstance(entities, dict):
        return False, "Entities field is not a JSON object"
    hallucinated = []
    for field in REQUIRED_ENTITY_FIELDS:
        value = entities.get(field)
        if value is None or value == "" or value == []:
            continue
        if not _value_in_input(value, input_text):
            hallucinated.append(f"{field}={json.dumps(value)}")
    if hallucinated:
        return False, f"Possible hallucinated entities: {'; '.join(hallucinated)}"
    return True, "All entities derivable from input"


def score_case(parsed, parse_error, input_text, gold_priority, gold_entities,
               response_coherent=None, audited=False):
    json_ok, json_reason = check_valid_json(parsed, parse_error)
    priority_ok, priority_reason = (False, "Skipped -- invalid JSON") if not json_ok else check_priority(parsed, gold_priority)
    entities_ok, entities_reason = (False, "Skipped -- invalid JSON") if not json_ok else check_entities(parsed, input_text, gold_entities)
    if not audited:
        response_ok = True
        response_reason = "Auto-pass (non-audited case)"
    elif not json_ok:
        response_ok = False
        response_reason = "Skipped -- invalid JSON"
    elif response_coherent is None:
        response_ok = True
        response_reason = "Pending judge evaluation"
    else:
        response_ok = response_coherent
        response_reason = "Judge: PASS" if response_coherent else "Judge: FAIL"
    all_pass = json_ok and priority_ok and entities_ok and response_ok
    return {
        "pass": all_pass,
        "criteria": {
            "json_valid": {"pass": json_ok, "reason": json_reason},
            "priority_correct": {"pass": priority_ok, "reason": priority_reason},
            "entities_accurate": {"pass": entities_ok, "reason": entities_reason},
            "response_coherent": {"pass": response_ok, "reason": response_reason},
        },
    }

# ---------------------------------------------------------------------------
# ジャッジプロンプトモジュール（judge_prompts.py から移植）
# ---------------------------------------------------------------------------

JUDGE_SYSTEM_PROMPT = """You are an eval judge for a support ticket processing system. Your job is to assess whether a drafted response appropriately addresses the support ticket given the priority classification.

Rules:
- The response must not contradict the priority classification
- The response must address the actual content of the ticket
- For feature requests classified as P4: the response must NOT promise to "fix" it or treat it as a bug
- For vague tickets: the response should ask for more information
- For multi-issue tickets: the response should acknowledge all issues mentioned
- Tone should be professional regardless of the ticket's tone

Reply with exactly one line: PASS or FAIL followed by a one-sentence reason.
Example: PASS - Response appropriately acknowledges the feature request and sets expectations.
Example: FAIL - Response promises to fix a feature request as if it were a bug."""


def judge_response(client, input_text, priority, response_text):
    user_message = f"""Evaluate this support ticket response:

SUPPORT TICKET:
{input_text}

ASSIGNED PRIORITY: {priority}

DRAFTED RESPONSE:
{response_text}

Does the response appropriately address the ticket without contradicting the classification? Reply PASS or FAIL with a one-sentence reason."""
    try:
        result = _call_api_with_retry(
            client, model=MODEL, max_tokens=150, system=JUDGE_SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_message}],
        )
        judge_output = result.content[0].text.strip()
        if judge_output.upper().startswith("PASS"):
            return True, judge_output
        elif judge_output.upper().startswith("FAIL"):
            return False, judge_output
        else:
            return True, f"Ambiguous judge output (defaulting to PASS): {judge_output}"
    except Exception as e:
        return True, f"Judge call failed (defaulting to PASS): {e}"

# ---------------------------------------------------------------------------
# コア評価ランナー
# ---------------------------------------------------------------------------

def _call_api_with_retry(client, **kwargs):
    """レート制限や過負荷エラーに対して指数バックオフでAPIコールをラップする。"""
    delays = [1, 2, 4]
    last_err = None
    for attempt in range(len(delays) + 1):
        try:
            return client.messages.create(**kwargs)
        except anthropic.RateLimitError as e:
            last_err = e
        except anthropic.APIStatusError as e:
            if e.status_code == 529:
                last_err = e
            else:
                raise
        if attempt < len(delays):
            print(f" (retry {attempt+1} in {delays[attempt]}s)", end="", flush=True)
            time.sleep(delays[attempt])
    raise last_err


def run_single_prompt(client, system_prompt, user_input):
    result = _call_api_with_retry(
        client, model=MODEL, max_tokens=MAX_TOKENS, system=system_prompt,
        messages=[{"role": "user", "content": user_input}],
    )
    return result.content[0].text


def run_chain(client, prompts, user_input):
    output = ""
    step_outputs = []
    for i, system_prompt in enumerate(prompts):
        if i == 0:
            user_message = user_input
        else:
            parts = [f"ORIGINAL TICKET:\n{user_input}"]
            for j, prev_output in enumerate(step_outputs):
                parts.append(f"STEP {j+1} OUTPUT:\n{prev_output}")
            user_message = "\n\n".join(parts)
        try:
            output = run_single_prompt(client, system_prompt, user_message)
        except Exception as e:
            output = json.dumps({"error": f"Chain step {i+1} failed: {e}"})
        step_outputs.append(output)
    return output


def run_eval(client, prompts, cases_data, verbose=False):
    cases = cases_data["cases"]
    categories = cases_data["categories"]
    total = len(cases)
    results = []
    print(f"{total}件の評価ケースを実行中...")
    for i, case in enumerate(cases):
        case_id = case["id"]
        print(f"  ケース {case_id:>2}/{total}...", end="", flush=True)
        try:
            if len(prompts) == 1:
                raw_output = run_single_prompt(client, prompts[0], case["input"])
            else:
                raw_output = run_chain(client, prompts, case["input"])
        except Exception as e:
            raw_output = f"API ERROR: {e}"
        parsed, parse_error = parse_output(raw_output)
        response_coherent = None
        if case.get("audited_response") and parsed:
            priority = parsed.get("priority", "")
            response_text = parsed.get("response", "")
            if response_text:
                judge_pass, judge_reason = judge_response(client, case["input"], priority, response_text)
                response_coherent = judge_pass
        result = score_case(
            parsed=parsed, parse_error=parse_error, input_text=case["input"],
            gold_priority=case["gold_priority"], gold_entities=case["gold_entities"],
            response_coherent=response_coherent, audited=case.get("audited_response", False),
        )
        result["case_id"] = case_id
        result["category"] = case["category"]
        result["raw_output"] = raw_output
        results.append(result)
        status = "PASS" if result["pass"] else "FAIL"
        if result["pass"]:
            print(f" {status}")
        else:
            failed_criteria = [
                name for name, crit in result["criteria"].items()
                if not crit["pass"]
            ]
            reasons = [
                f'{name}: {result["criteria"][name]["reason"]}'
                for name in failed_criteria
            ]
            print(f" {status} [{"; ".join(reasons)}]")
    category_results = {}
    for cat_key, cat_info in categories.items():
        cat_case_ids = cat_info["case_ids"]
        cat_results = [r for r in results if r["case_id"] in cat_case_ids]
        passed = sum(1 for r in cat_results if r["pass"])
        category_results[cat_key] = {
            "label": cat_info["label"],
            "passed": passed,
            "total": len(cat_results),
        }
    total_passed = sum(1 for r in results if r["pass"])
    return {
        "results": results, "categories": category_results,
        "total_passed": total_passed, "total_cases": total,
    }

# ---------------------------------------------------------------------------
# 可視化
# ---------------------------------------------------------------------------

def _score_color(pct):
    if pct >= 80: return "#22c55e"
    if pct >= 60: return "#eab308"
    return "#ef4444"


def _display_results(eval_result, label="Current"):
    global _baseline_score, _baseline_total
    total_passed = eval_result["total_passed"]
    total_cases = eval_result["total_cases"]
    pct = round(100 * total_passed / total_cases) if total_cases else 0
    color = _score_color(pct)

    baseline_html = ""
    if _baseline_score is not None:
        bl_pct = round(100 * _baseline_score / _baseline_total)
        diff = total_passed - _baseline_score
        diff_str = f"+{diff}" if diff >= 0 else str(diff)
        diff_color = "#22c55e" if diff > 0 else ("#ef4444" if diff < 0 else "#6b7280")
        baseline_html = f'<div style="font-size:16px;color:#6b7280;margin-top:4px;">Baseline: {_baseline_score}/{_baseline_total} ({bl_pct}%) &rarr; Current: {total_passed}/{total_cases} ({pct}%) <span style="color:{diff_color};font-weight:bold;">[{diff_str}]</span></div>'

    html = f'''
    <div style="background:#f8fafc;border:2px solid {color};border-radius:12px;padding:24px;margin:16px 0;font-family:system-ui,-apple-system,sans-serif;">
      <div style="text-align:center;">
        <div style="font-size:14px;color:#6b7280;text-transform:uppercase;letter-spacing:1px;">{label}</div>
        <div style="font-size:64px;font-weight:bold;color:{color};">{total_passed}/{total_cases}</div>
        <div style="font-size:24px;color:{color};">({pct}%)</div>
        {baseline_html}
      </div>
    </div>
    '''
    display(HTML(html))

    cats = eval_result["categories"]
    labels = [cats[k]["label"] for k in cats]
    passed = [cats[k]["passed"] for k in cats]
    totals = [cats[k]["total"] for k in cats]
    failed = [t - p for t, p in zip(totals, passed)]

    fig, ax = plt.subplots(figsize=(10, 3.5))
    y_pos = range(len(labels))
    ax.barh(y_pos, passed, color='#22c55e', label='Passed', height=0.6)
    ax.barh(y_pos, failed, left=passed, color='#fee2e2', label='Failed', height=0.6)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=12)
    ax.set_xlabel('Cases', fontsize=12)
    ax.set_title('Score by Category', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right')
    for i, (p, t) in enumerate(zip(passed, totals)):
        ax.text(t + 0.1, i, f'{p}/{t}', va='center', fontsize=11, fontweight='bold')
    ax.set_xlim(0, max(totals) + 1.5)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()


def _log_run(label, eval_result):
    _iteration_log.append({
        "timestamp": datetime.now().strftime("%H:%M:%S"),
        "label": label,
        "passed": eval_result["total_passed"],
        "total": eval_result["total_cases"],
        "pct": round(100 * eval_result["total_passed"] / eval_result["total_cases"]),
        "categories": {k: v.copy() for k, v in eval_result["categories"].items()},
    })

# ---------------------------------------------------------------------------
# パブリックAPI -- チームが使用する関数
# ---------------------------------------------------------------------------

def run_eval_suite(prompt_or_prompts, label="Current"):
    """完全な評価スイートを実行し、視覚的な結果を表示する。

    Args:
        prompt_or_prompts: 単一のシステムプロンプト文字列、
                          またはチェーンモード用のプロンプト文字列のリスト。
        label: この実行のラベル（結果とイテレーションログに表示）。
    """
    global _baseline_score, _baseline_total

    client = anthropic.Anthropic()
    cases_data = _load_cases(_EVAL_CASES_B64)

    if isinstance(prompt_or_prompts, str):
        prompts = [prompt_or_prompts]
        mode = "single prompt"
    else:
        prompts = list(prompt_or_prompts)
        mode = f"{len(prompts)}-step chain"

    print(f"Mode: {mode}")
    print(f"Model: {MODEL}")
    print()

    start_time = time.time()
    eval_result = run_eval(client, prompts, cases_data)
    elapsed = time.time() - start_time

    print(f"\n完了（{elapsed:.1f}秒）")

    if _baseline_score is None:
        _baseline_score = eval_result["total_passed"]
        _baseline_total = eval_result["total_cases"]
        print(f"ベースライン設定: {_baseline_score}/{_baseline_total}")

    _log_run(label, eval_result)
    _display_results(eval_result, label=label)
    return eval_result


def compare_prompts(prompt_a, prompt_b, label_a="Variant A", label_b="Variant B"):
    """2つのプロンプトバリアントを実行し、並べて比較表示する."""
    client = anthropic.Anthropic()
    cases_data = _load_cases(_EVAL_CASES_B64)

    prompts_a = [prompt_a] if isinstance(prompt_a, str) else list(prompt_a)
    prompts_b = [prompt_b] if isinstance(prompt_b, str) else list(prompt_b)

    print(f"{label_a} を実行中...")
    result_a = run_eval(client, prompts_a, cases_data)
    print(f"\n{label_b} を実行中...")
    result_b = run_eval(client, prompts_b, cases_data)

    _log_run(label_a, result_a)
    _log_run(label_b, result_b)

    pct_a = round(100 * result_a["total_passed"] / result_a["total_cases"])
    pct_b = round(100 * result_b["total_passed"] / result_b["total_cases"])

    html = f'''
    <div style="display:flex;gap:16px;margin:16px 0;font-family:system-ui,-apple-system,sans-serif;">
      <div style="flex:1;background:#f8fafc;border:2px solid {_score_color(pct_a)};border-radius:12px;padding:20px;text-align:center;">
        <div style="font-size:13px;color:#6b7280;text-transform:uppercase;">{label_a}</div>
        <div style="font-size:48px;font-weight:bold;color:{_score_color(pct_a)};">{result_a["total_passed"]}/{result_a["total_cases"]}</div>
        <div style="font-size:20px;color:{_score_color(pct_a)};">({pct_a}%)</div>
      </div>
      <div style="flex:1;background:#f8fafc;border:2px solid {_score_color(pct_b)};border-radius:12px;padding:20px;text-align:center;">
        <div style="font-size:13px;color:#6b7280;text-transform:uppercase;">{label_b}</div>
        <div style="font-size:48px;font-weight:bold;color:{_score_color(pct_b)};">{result_b["total_passed"]}/{result_b["total_cases"]}</div>
        <div style="font-size:20px;color:{_score_color(pct_b)};">({pct_b}%)</div>
      </div>
    </div>
    '''
    display(HTML(html))

    cats_a = result_a["categories"]
    cats_b = result_b["categories"]
    cat_keys = list(cats_a.keys())
    labels = [cats_a[k]["label"] for k in cat_keys]
    scores_a = [cats_a[k]["passed"] for k in cat_keys]
    scores_b = [cats_b[k]["passed"] for k in cat_keys]
    totals = [cats_a[k]["total"] for k in cat_keys]

    fig, ax = plt.subplots(figsize=(10, 4))
    y = range(len(labels))
    bar_height = 0.35
    ax.barh([i - bar_height/2 for i in y], scores_a, bar_height, label=label_a, color='#3b82f6')
    ax.barh([i + bar_height/2 for i in y], scores_b, bar_height, label=label_b, color='#f97316')
    ax.set_yticks(list(y))
    ax.set_yticklabels(labels, fontsize=12)
    ax.set_xlabel('Cases Passed', fontsize=12)
    ax.set_title('Category Comparison', fontsize=14, fontweight='bold')
    ax.legend(loc='lower right')
    for i, (sa, sb, t) in enumerate(zip(scores_a, scores_b, totals)):
        ax.text(max(sa, sb) + 0.15, i, f'/{t}', va='center', fontsize=10, color='#6b7280')
    ax.set_xlim(0, max(totals) + 1.5)
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

    rows = ""
    for k in cat_keys:
        sa = cats_a[k]["passed"]
        sb = cats_b[k]["passed"]
        t = cats_a[k]["total"]
        diff = sb - sa
        winner = label_b if diff > 0 else (label_a if diff < 0 else "Tie")
        diff_color = "#22c55e" if diff > 0 else ("#ef4444" if diff < 0 else "#6b7280")
        diff_str = f"+{diff}" if diff > 0 else str(diff)
        rows += f'<tr><td>{cats_a[k]["label"]}</td><td>{sa}/{t}</td><td>{sb}/{t}</td><td style="color:{diff_color};font-weight:bold;">{diff_str}</td><td>{winner}</td></tr>'

    table_html = f'''
    <table style="border-collapse:collapse;width:100%;margin:16px 0;font-family:system-ui,-apple-system,sans-serif;">
    <thead><tr style="background:#f1f5f9;"><th style="padding:8px;text-align:left;">Category</th><th>{label_a}</th><th>{label_b}</th><th>Delta</th><th>Winner</th></tr></thead>
    <tbody>{rows}</tbody>
    </table>
    '''
    display(HTML(table_html))
    return result_a, result_b


def show_iteration_log():
    """イテレーション履歴を表と折れ線グラフで表示する."""
    if not _iteration_log:
        print("まだイテレーションが記録されていません。先に run_eval_suite() を実行してください。")
        return

    rows = ""
    for i, entry in enumerate(_iteration_log):
        color = _score_color(entry["pct"])
        rows += f'<tr><td>{i+1}</td><td>{entry["timestamp"]}</td><td>{entry["label"]}</td><td style="color:{color};font-weight:bold;">{entry["passed"]}/{entry["total"]} ({entry["pct"]}%)</td></tr>'

    table_html = f'''
    <h3 style=\"font-family:system-ui,-apple-system,sans-serif;\">イテレーションログ</h3>
    <table style="border-collapse:collapse;width:100%;margin:8px 0;font-family:system-ui,-apple-system,sans-serif;">
    <thead><tr style="background:#f1f5f9;"><th style="padding:8px;text-align:left;">#</th><th>Time</th><th>Label</th><th>Score</th></tr></thead>
    <tbody>{rows}</tbody>
    </table>
    '''
    display(HTML(table_html))

    if len(_iteration_log) > 1:
        fig, ax = plt.subplots(figsize=(10, 4))
        x = range(1, len(_iteration_log) + 1)
        pcts = [e["pct"] for e in _iteration_log]
        ax.plot(x, pcts, 'o-', color='#3b82f6', linewidth=2, markersize=8)
        ax.axhline(y=75, color='#22c55e', linestyle='--', alpha=0.7, label='Pass threshold (75%)')
        ax.axhline(y=90, color='#eab308', linestyle='--', alpha=0.7, label='Optimizer tier (90%)')
        for i, (xi, pct) in enumerate(zip(x, pcts)):
            ax.annotate(f'{pct}%', (xi, pct), textcoords="offset points",
                        xytext=(0, 12), ha='center', fontsize=10, fontweight='bold')
        ax.set_xlabel('Iteration', fontsize=12)
        ax.set_ylabel('Pass Rate (%)', fontsize=12)
        ax.set_title('Score Progression', fontsize=14, fontweight='bold')
        ax.set_ylim(0, 105)
        ax.set_xticks(list(x))
        ax.set_xticklabels([e["label"][:20] for e in _iteration_log], rotation=30, ha='right', fontsize=9)
        ax.legend(loc='lower right')
        plt.tight_layout()
        plt.show()


print("評価ハーネスを読み込みました。")
print()
print("利用可能な関数:")
print("  run_eval_suite(prompt_or_prompts, label='...')  -- 完全な評価を実行し、視覚的な結果を表示")
print("  compare_prompts(prompt_a, prompt_b, ...)        -- 2つのバリアントのA/B比較")
print("  show_iteration_log()                            -- 全実行履歴とスコア進捗グラフを表示")


In [ ]:
# ============================================================
# 最初の評価を実行 -- ベースラインスコアを確立する
# ============================================================

results = run_eval_suite(system_prompt, label="Baseline (Broken Prompt)")

---

### 根本原因を特定する前に: Claude で診断を実行する

Claude はここで診断ツールとして役立ちます。何が問題かを決める前に、新しい Claude の会話を開いて5件の失敗例（セル4）をこのプロンプトと一緒に貼り付けてください:

> "Here are 5 cases where a support ticket classifier is failing. What structural patterns do you see? What are the most likely root causes?"

出力をプロファイラーの結果のように扱ってください — 評価すべき仮説のセットであり、受け入れるべき判決ではありません。Claude が特定したものに同意しますか？見逃していたものはありますか？

そして先に進む前に自分の診断を明示してください。

---

## スコアのベンチマーク

| ステージ | 期待スコア | 意味 |
|-------|---------------|---------------|
| ベースライン（壊れたプロンプト） | ~4/21 (19%) | 出発点 — 改善の余地が大きい |
| 良好なシングルプロンプト修正 | 9–12/21 (43–57%) | 主要な問題に対処できている |
| 強力なチェーンソリューション | 16–19/21 (76–90%) | 優秀 — 技術をマスターした |
| リファレンスソリューション | 21/21 (100%) | ゴールドスタンダード |

ベースラインが低くても心配しないでください — それは意図的なものです！目標は繰り返して改善することです。

## 診断ガイダンス

修正を始める前に、*何が問題か*を理解することに10〜15分かけてください。実際の案件のように扱ってください — 処方する前に診断する。

### どこに注目するか

ベースライン評価の **カテゴリ別内訳** を見てください:
- スコアが0のカテゴリはどれですか？そこが最もレバレッジの高いターゲットです。
- すでにパスしているカテゴリはどれですか？それらを保護してください — 難しいケースの修正が簡単なケースを壊すことがよくあります。

### 失敗例で注目すべきパターン

1. **タスクの干渉** — プロンプトはモデルに分類、抽出、*そして*同時に回答のドラフトを依頼しています。回答ドラフトタスクが共感的な言語を活性化させると、優先度分類が膨らむことがあります。抽出で何も見つからないと、モデルはフィールドを埋めるためにエンティティを作り出します。

2. **幻覚エンティティ** — プロンプトは「空でもすべての JSON フィールドを含めよ」と言っています。モデルは「含める」を「null のまま残す」ではなく「埋める」と解釈します。曖昧なチケットが偽造されたプロダクト名やエラーコードを引き起こします。

3. **感情的なトーンによる優先度の偏り** — プロンプトは P1 を「システム障害、全ユーザーが影響」と定義していますが、*トーンを無視してコンテンツに集中せよ*とは言っていません。怒りや緊急のような言葉は、実際の問題が機能要望であっても優先度を上げてしまいます。

4. **機能要望の誤分類** — 緊急に聞こえる機能要望でも P4 だという明示的な例がないと、モデルは緊急を示す言葉を高い優先度レベルにデフォルトでマッピングします。

### メタプロンプティングを試す

壊れたプロンプトを2〜3件の失敗例と一緒に新しい Claude の会話に貼り付けてください。こう聞いてみましょう:

> "What structural issues in this prompt could cause these outputs? Be specific about which instructions are ambiguous, conflicting, or missing."

これはツールキットの中で最も強力な診断ツールの1つです — Claude に自身のプロンプトを分析させてください。

---

### どの技術が適切か分からない？Claude に聞いてみましょう。

診断した失敗パターンを取り上げて Claude に聞いてください:

> "I'm fixing a prompt that fails on [describe your failure mode]. Which of these would you recommend: meta-prompting, prompt chaining, few-shot examples, or constraint sharpening? Give me your reasoning for this specific failure pattern."

そして自分で判断してください。重要なのは Claude が正しいかどうかではなく — その推論が自分の診断に対して成立するかどうかです。

---

## ツールキット

4つの技術、それぞれ異なる失敗モードに適しています。診断に基づいて選んでください — ルーブリックは技術の数ではなく*根拠*を評価します。

### 1. メタプロンプティング
**何か:** Claude を使ってプロンプト自体を分析・改善する。

**いつ使うか:** プロンプトが*なぜ*失敗するか分からないとき、または体系的に曖昧さを見つけ出したいとき。

**方法:** 壊れたプロンプトと失敗例を Claude に貼り付けて、構造的な問題の特定、書き直しの提案、または改善された指示の生成を依頼してください。

---

### 2. プロンプトチェーニング
**何か:** モノリシックなプロンプトを、各ステップが集中した責務を持つマルチステップのパイプラインに分解する。

**いつ使うか:** プロンプトが同時に多くのことをしすぎていて、タスクが互いに干渉しているとき（モデルが同時に回答をドラフトしているときに分類精度が低下する）。

**方法:** 優先度分類 -> エンティティ抽出 -> 回答のようにステップに分割します。各ステップに独自のシステムプロンプトを持たせます。評価ハーネスはチェーンモードをサポートしています — プロンプトのリストを渡してください。

**トレードオフ:** 各チェーンステップは独立したAPIコールです。顧客のレイテンシ予算は5秒なので、3回の順次コールはそれぞれ〜1.5秒で完了する必要があります。

---

### 3. A/Bテスト
**何か:** 同じ評価セットに対して競合するプロンプトバリアントを実行し、定量的に比較する。

**いつ使うか:** どのフレーミング、指示セット、またはアーキテクチャがより優れているか不確かなとき。

**方法:** `compare_prompts(prompt_a, prompt_b)` を使って両方のバリアントを実行し、サイドバイサイドのカテゴリ別内訳を確認してください。

---

### 4. 自動プロンプト最適化
**何か:** 体系的なイテレーションループ — バリアントを生成し、それぞれを評価し、最良のものを選択する。

**いつ使うか:** 行き詰まって体系的に解決空間を探索する必要があるとき。

**方法:** プロンプトバリアントを（メタプロンプティングを使って）生成し、それぞれを評価し、最高のパフォーマーを保持するループを書いてください。手動版: 最も難しい指示の3つのバリアントを試して、すべて評価し、勝者を残す。

In [ ]:
# ============================================================
# チェーンモード -- 複数ステップに分割する場合はこちらを使用
# ============================================================

# ステップ1: 分類器
step1_prompt = """
[分類器のプロンプトをここに記述 -- 優先度分類のみに集中する]
"""

# ステップ2: エンティティ抽出器
step2_prompt = """
[エンティティ抽出のプロンプトをここに記述 -- product、version、error codes、affected users を抽出]
"""

# ステップ3: 応答ドラフター
step3_prompt = """
[応答ドラフトのプロンプトをここに記述 -- 顧客への返答をドラフトし最終JSONを組み立てる]
"""

# チェーンを実行する（準備ができたらコメントを外す）
# results = run_eval_suite([step1_prompt, step2_prompt, step3_prompt], label="Chain v1")

In [ ]:
# ============================================================
# A/B 比較 -- 2つのプロンプトバリアントを比較する
# ============================================================

prompt_a = system_prompt  # 現在のベストプロンプト

prompt_b = """
[バリアントをここに貼り付ける]
"""

# 比較を実行する（準備ができたらコメントを外す）
# result_a, result_b = compare_prompts(prompt_a, prompt_b, label_a="Current Best", label_b="New Variant")

In [ ]:
# ============================================================
# イテレーションログ -- 実行ごとの進捗を追跡する
# ============================================================

# 評価ハーネスは各実行を自動的にログに記録します。
# イテレーション履歴とスコア進捗を確認するにはこれを呼び出してください:

show_iteration_log()

---

### 行き詰まった？ファシリテーターに聞く前に Claude に聞いてみましょう。

試したことと何が起きたかを Claude に具体的に伝えてください:

> "I'm iterating on a prompt. I tried [describe your change]. My score went from X to Y. The categories still failing are [list them]. What would you try next and why?"

答えを求めているのではありません — 診断を実行しているのです。行動する前に、自分の診断に対して提案を評価してください。

---

## 振り返り — *あなた*は何をしたか vs. Claude は何をしたか？

ここが本当の学びが起きる場所です — そして優秀なパートナーエンジニアが自分を際立たせる場所でもあります。

スコアを共有する前に、5分かけてこれらの質問に正直に答えてください:

### 1. どこであなたのプロンプトエンジニアリングが機能したか？
どの改善が、あなたが意図的に行った構造的な決定から来ているか — チェーンステップ、明示的な制約、few-shot の例、null 処理の指示など？

2〜3つの具体的な例を書いてください: *「Y の失敗モードを診断したので X を追加した。」*

### 2. どこで Claude の根本的な能力が機能したか？
一部の改善は Claude がすでに得意なことによって起こります — あなたは邪魔にならないようにしただけです。その違いについて正直に答えてください。*「すべてのフィールドを埋めることを強制するのをやめたら、Claude は英語が母国語でない人のものを上手く扱えた。」*

### 3. 驚いたことは？
期待通りに機能しなかったことは？Claude が予測より良く — あるいは悪く — 行動したのはどんな時でしたか？

### 4. CTO に何を伝えるか？
技術的な意思決定者にアーキテクチャを90秒で説明するとしたら、何を言いますか？ここで共有する前にドラフトを作ってください。

---

## Claude から個人分析を受け取る

振り返りが終わったら、最終スコアと回答を新しい Claude の会話に貼り付けてこのプロンプトを使ってください:

```
I just completed a prompt engineering exercise. Here's my summary:

Final score: [X/21 — Y%]
Baseline: [X/21 — Y%]

Techniques I used:
[List them + brief rationale for each]

Where my engineering did the work:
[Your answer from question 1]

Where Claude's capability did the work:
[Your answer from question 2]

What surprised me:
[Your answer from question 3]

Please give me a candid analysis:
1. Where did my prompt engineering approach shine?
2. Where did I rely on Claude's capability when I should have engineered more explicitly?
3. What would make my approach more defensible in an enterprise context?
4. What's one technique I didn't use that would likely push my score higher?
```

---

## ストレッチ: プロセス監査

*高スコアを達成した速い人向け。オプションです。*

これが方法論とデモを分ける問いです: **あなたが仕事をしたのか、Claude がしたのか？**

### ステップ1: Claude にプロセスを監査してもらう

新しい Claude の会話を開いてこれを送ってください:

```
Here's my original prompt:
[paste the broken prompt from Cell 3]

Here's my final prompt (or chain):
[paste your best version]

Score progression: [paste from your iteration log]

How much of this improvement came from diagnostic decisions I made — and how much would have happened if you'd just rewritten the prompt from scratch without the eval loop?
```

### ステップ2: 実験を実行する

方法論なしに Claude に元のプロンプトを直接修正してもらいましょう — ただこれだけで: *"Here's a broken support ticket classifier. Rewrite it to fix the failures. Return only the improved prompt."*

Claude の書き直しを下のセルに貼り付けて実行してください。

### ステップ3: 比較する

同じスコア？違うスコア？どちらにしても、本当の問いは: 来週別の壊れたプロンプトで繰り返せる形で、なぜあなたのアプローチが機能するかを説明できるか？

説明できるスコアが方法論です。説明できないスコアはデモです。

---

In [ ]:
# ============================================================
# プロセス監査 -- あなたのアプローチと Claude の直接修正を比較する
# ============================================================

claude_fix = """
[Claude の直接書き直しをここに貼り付ける]
"""

results_audit = run_eval_suite(claude_fix, label="Claude direct fix")

## エンタープライズバイヤーへの提示方法

この演習の目標は単により良いスコアを得ることではありません — 技術的なバイヤーを実際のプロンプトエンジニアリング案件に通り抜けさせる自信を築くことです。その方法はこちらです。

### ナラティブの流れ

| ビート | 言うこと | なぜ機能するか |
|------|-------------|---------------|
| **問題** | "あなたのプロンプトは同時に多くのことをしすぎていて、タスクが互いに干渉しています。" | 症状ではなく根本原因を診断している。 |
| **診断** | "Claude を使って自身の失敗モードを分析しました — 特定したものがこちらです。" | メタプロンプティングの実践を示している。バイヤーはこれに反応する。 |
| **修正** | "[N]ステップのチェーンに分解しました: 最初に分類、次にエンティティ抽出、最後に回答。各ステップに集中しており、他を汚染できません。" | アーキテクチャ的な思考を示している（単なるプロンプトの微調整ではなく）。 |
| **証明** | "修正前: パス率19%。修正後: 86%。カテゴリ別内訳がここにあります — 機能要望が最も改善しました、なぜならトーンではなくコンテンツを評価するようになったからです。" | 定量的。具体的。説明可能。 |
| **予防** | "チームが本番出荷前にすべての新しいプロンプトに対して実行できる3問チェックリストがここにあります。" | 使えるものを残している。 |

### 良いと優秀を分けるもの

- **良い:** "プロンプトを修正してスコアが上がった。"
- **優秀:** "構造的な失敗モードを診断し、それに特化した技術を選択し、各失敗カテゴリにわたるデルタを示すことができる。"

第二のバージョンがエンタープライズの案件を成約させます。

### 絶対に言ってはいけない一言

"ただ Claude に修正してもらった" とは絶対に言わないでください。メタプロンプティングが役立ったとしても、診断の判断は自分のものとして主張してください。何を尋ねるかを決めたのはあなたです。結果を評価したのはあなたです。どの修正を出荷するかを判断したのはあなたです。それがあなたがもたらす価値です。